# Investigation of rare non-synonymous variants in RAB32

- Project: RAB GTPase gene investigation and burden analysis across populations with Parkinson’s disease
- Version: Python/3.10.19
- Last Updated: 13-OCT-2026

# Notebook Overview
- 1 RAB32 - Single Gene Analysis in GP2 WGS data
  - 1.1 Loading Python libraries
  - 1.2 Set Paths
  - 1.3 Defining functions
  - 1.4 Make Working Directory
  - 1.5 Annotation of the gene
    - 1.5.1 Extract the region using PLINK
    - 1.5.2 Download reference fasta and prepare vcf for annotation
  - 1.6 Annotate using ANNOVAR
  - 1.7 RAB32 missense variant analysis
    - 1.7.1 Look into RAB32 variants of interest

# *RAB32* - Single Gene Analysis in GP2 WGS data

## Loading Python libraries

In [ ]:
# Use the os package to interact with the environment
import os

# Bring in Pandas for Dataframe functionality
import pandas as pd

# Numpy for basics
import numpy as np

# Use StringIO for working with file contents
from io import StringIO

# Enable IPython to display matplotlib graphs
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Import the iPython HTML rendering for displaying links to Google Cloud Console
from IPython.display import display, HTML

# Import urllib modules for building URLs to Google Cloud Console
import urllib.parse

# BigQuery for querying data
from google.cloud import bigquery

# Use pathlib for file path manipulation
import pathlib
from pathlib import Path

# Import Sys
import sys as sys

# Import Seaborn
import seaborn as sns

# Install plotnine: a ggplot2-compatible Python plotting package
from plotnine import *

# Import Glob
import glob

# Import Re
import re

# Import Subprocess
import subprocess

# Import CYVCF2
#from cyvcf2 import VCF

# Import tools for meta-analysis
from scipy import stats
from scipy.stats import fisher_exact

# Import collections
from collections import defaultdict, deque

# Import Tempfile
import tempfile

# Import Shutil
import shutil

# Always show all columns in a Pandas DataFrame
pd.set_option('display.max_columns', None)
pd.set_option('future.no_silent_downcasting', True)

## Set paths

In [ ]:
## GP2 v12.0
REL12_PATH = pathlib.Path(pathlib.Path.home(), 'workspace/gp2_tier2_eu_release12')
WGS_DATA_PATH = pathlib.Path(REL12_PATH, 'wgs/dragen_joint_calling/plink')
WGS_RELATED_PATH = pathlib.Path(REL12_PATH, 'wgs/dragen_joint_calling/related_samples')
CLINICAL_DATA_PATH = pathlib.Path(REL12_PATH, 'clinical_data/master_key_release12_final_vwb.csv')

## Defining functions

In [ ]:
# Utility routine for printing a shell command before executing it
def shell_do(command):
    print(f'Executing: {command}', file=sys.stderr)
    !$command
    
def shell_return(command):
    print(f'Executing: {command}', file=sys.stderr)
    output = !$command
    return '\n'.join(output)

# Utility routine for printing a query before executing it
def bq_query(query):
    print(f'Executing: {query}', file=sys.stderr)
    return pd.read_gbq(query, project_id=BILLING_PROJECT_ID, dialect='standard')

# Utility routine for display a message and a link
def display_html_link(description, link_text, url):
    html = f'''
    <p>
    </p>
    <p>
    {description}
    <a target=_blank href="{url}">{link_text}</a>.
    </p>
    '''

    display(HTML(html))

## Make Working Directory

In [ ]:
%%bash
mkdir /home/jupyter/results/RAB32/

for ancestry in {'AAC','AFR','AJ','AMR','CAS','EAS','EUR','FIN','MDE','SAS','CAH'} ;
do
# Make a folder for each ancestry
mkdir /home/jupyter/results/RAB32/RAB32_WGS_"$ancestry"
done

## Annotation of the gene

### Extract the region using PLINK

- Extract RAB32 gene
- RAB32 coordinates: Chromosome 6: 146,543,833-146,554,953

In [ ]:
# Basic QC before extracting
ancestries = {'AAC','AFR','AJ','AMR','CAS','EAS','EUR','FIN','MDE','SAS','CAH'}
WGS_DATA_PATH = f'/home/jupyter/workspace/gp2_tier2_eu_release12/wgs/dragen_joint_calling/plink'
for ancestry in ancestries:
    
    WORK_DIR = f'/home/jupyter/results/RAB32/RAB32_WGS_{ancestry}'
    
    ! /home/jupyter/tools/plink/plink2 \
    --pfile {WGS_DATA_PATH}/{ancestry}/chr6_{ancestry}_release12  \
    --geno 0.05 \
    --make-pgen \
    --max-alleles 2 \
    --out {WORK_DIR}/chr6.qc_{ancestry}_release12

In [ ]:
# Extract region using plink

for ancestry in ancestries:
    
    WORK_DIR = f'/home/jupyter/results/RAB32/RAB32_WGS_{ancestry}'
    ! /home/jupyter/tools/plink/plink2 \
    --pfile {WORK_DIR}/chr6.qc_{ancestry}_release12 \
    --chr 6 \
    --from-bp 146543833 \
    --to-bp 146554953 \
    --make-pgen \
    --out {WORK_DIR}/RAB32

    !rm {WORK_DIR}/chr6.qc_{ancestry}_release12*

In [ ]:
# Turn binary files into VCF
for ancestry in ancestries:
    
    WORK_DIR = f'/home/jupyter/results/RAB32/RAB32_WGS_{ancestry}'
    
    ! /home/jupyter/tools/plink/plink2 \
    --pfile {WORK_DIR}/RAB32 \
    --recode vcf id-paste=iid \
    --out {WORK_DIR}/{ancestry}.RAB32

### Download reference fasta and prepare vcf for annotation

In [ ]:
%%bash
cd /home/jupyter/tools/reference/
wget http://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/chr6.fa.gz
gunzip chr6.fa.gz
samtools faidx chr6.fa

In [ ]:
%%bash
for ancestry in "AAC" "AFR" "AJ" "AMR" "CAS" "EAS" "EUR" "FIN" "MDE" "SAS" "CAH"; do
    WORK_DIR="/home/jupyter/results/RAB32/RAB32_WGS_$ancestry"
    awk '/^[^#]/{ $0 = "chr"$0 }1' $WORK_DIR/$ancestry.RAB32.vcf | sed 's/##contig=<ID=6/##contig=<ID=chr6/' > $WORK_DIR/$ancestry.RAB32.prep.vcf
done

In [ ]:
%%bash
for ancestry in "AAC" "AFR" "AJ" "AMR" "CAS" "EAS" "EUR" "FIN" "MDE" "SAS" "CAH"; do
    WORK_DIR="/home/jupyter/results/RAB32/RAB32_WGS_$ancestry"
    bcftools norm -m-both -o $WORK_DIR/$ancestry.RAB32.norm.vcf $WORK_DIR/$ancestry.RAB32.prep.vcf
    bcftools norm -f /home/jupyter/tools/reference/chr6.fa -o $WORK_DIR/$ancestry.RAB32.adapt.vcf $WORK_DIR/$ancestry.RAB32.norm.vcf
done

## Annotate using ANNOVAR

In [ ]:
# Annotate using ANNOVAR
ancestries = {'AAC','AFR','AJ','AMR','CAS','EAS','EUR','FIN','MDE','SAS','CAH'}
for ancestry in ancestries:
    
    WORK_DIR = f'/home/jupyter/results/RAB32/RAB32_WGS_{ancestry}'
    
    ! perl /home/jupyter/tools/annovar/table_annovar.pl {WORK_DIR}/{ancestry}.RAB32.adapt.vcf /home/jupyter/tools/annovar/humandb/ -buildver hg38 \
    -out {WORK_DIR}/{ancestry}.RAB32.annovar \
    -remove -protocol refGene,avsnp151,dbnsfp54a,clinvar_20260829,gnomad41_genome\
    -operation g,f,f,f,f \
    -polish \
    -nastring . \
    -vcfinput

In [ ]:
# Make lists of variants to keep - all coding, coding nonsynonymous (missense - as they are coded in ANNOVAR), deleterious (CADD_phred > 20)

for ancestry in ancestries:
    
    WORK_DIR = f'/home/jupyter/results/RAB32/RAB32_WGS_{ancestry}'
    print(f'WORKING ON: {ancestry}')
    
    # Read in ANNOVAR multianno file
    gene = pd.read_csv(f'{WORK_DIR}/{ancestry}.RAB32.annovar.hg38_multianno.txt', sep='\t')
    
    # Filter for the correct gene name (sometimes other genes are also included)
    gene = gene[gene['Gene.refGene'] == 'RAB32']

    # To filter by CADD_phred scores we first need to convert the CADD_phred column to numeric
    # Also replace the . values with NAs otherwise the string values cannot be converted
    gene['CADD_phred'] = gene['CADD_phred'].replace('.', np.nan)

    # Now convert to numeric
    gene['CADD_phred'] = pd.to_numeric(gene['CADD_phred'], errors='coerce')
    
    # Print number of variants in the different categories
    results = []

    utr5 = gene[gene['Func.refGene'] == 'UTR5']
    intronic = gene[gene['Func.refGene'] == 'intronic']
    exonic = gene[gene['Func.refGene'] == 'exonic']
    utr3 = gene[gene['Func.refGene'] == 'UTR3']
    coding_nonsynonymous = gene[(gene['Func.refGene'] == 'exonic') & (gene['ExonicFunc.refGene'] != 'synonymous SNV')]
    coding_synonymous = gene[(gene['Func.refGene'] == 'exonic') & (gene['ExonicFunc.refGene'] == 'synonymous SNV')]
    deleterious = gene[(gene['CADD_phred'] > 20)]

    print({ancestry})
    print('Total variants: ', len(gene))
    print('Intronic: ', len(intronic))
    print('UTR3: ', len(utr3))
    print('UTR5: ', len(utr5))
    print('Total exonic: ', len(exonic))
    print('  Synonymous: ', len(coding_synonymous))
    print('  Nonsynonymous: ', len(coding_nonsynonymous))
    print('Deleterious: ', len(deleterious))
    results.append((gene, intronic, utr3, utr5, exonic, coding_synonymous, coding_nonsynonymous, deleterious))
    print('\n')
    
    # Save in PLINK format - coding nonsynonymous
    variants_toKeep = coding_nonsynonymous[['Chr', 'Start', 'End', 'Gene.refGene']].copy()
    variants_toKeep.to_csv(f'{WORK_DIR}/{ancestry}_RAB32.nonsynonymous.variantstoKeep.txt', sep="\t", index=False, header=False)

    # Save in PLINK format - all coding variants
    variants_toKeep2 = exonic[['Chr', 'Start', 'End', 'Gene.refGene']].copy()
    variants_toKeep2.to_csv(f'{WORK_DIR}/{ancestry}_RAB32.all_coding.variantstoKeep.txt', sep="\t", index=False, header=False)

    # Save in PLINK format - deleterious variants
    variants_toKeep3 = deleterious[['Chr', 'Start', 'End', 'Gene.refGene']].copy()
    variants_toKeep3.to_csv(f'{WORK_DIR}/{ancestry}_RAB32.deleterious.variantstoKeep.txt', sep="\t", index=False, header=False)

## *RAB32* missense variant analyis

In [ ]:
for ancestry in ancestries:
    
    WORK_DIR = f'/home/jupyter/results/RAB32/RAB32_WGS_{ancestry}'
    
    ! /home/jupyter/tools/plink/plink2 \
    --pfile {WORK_DIR}/RAB32 \
    --make-bed \
    --out {WORK_DIR}/RAB32
    
    WORK_DIR = f'/home/jupyter/results/RAB32/RAB32_WGS_{ancestry}'
    
    ! /home/jupyter/tools/plink/plink1.9 \
    --bfile {WORK_DIR}/RAB32 \
    --extract range {WORK_DIR}/{ancestry}_RAB32.nonsynonymous.variantstoKeep.txt \
    --recode A \
    --out {WORK_DIR}/{ancestry}_RAB32.nonsynonymous.nonadj

In [ ]:
# Prepare overviews of non-synonymous variants in each ancestry
ancestries_nonsyn =  ['AAC','AFR','AJ','AMR','CAS','EAS','EUR', 'FIN','MDE','SAS','CAH']

for ancestry in ancestries_nonsyn:
    
    WORK_DIR = f'/home/jupyter/results/RAB32/RAB32_WGS_{ancestry}'
    
    print(f'WORKING ON: {ancestry}')

    # Read in plink recoded data (.raw file)
    recode = pd.read_csv(f'{WORK_DIR}/{ancestry}_RAB32.nonsynonymous.nonadj.raw', sep='\s+')

    # Make a list from the column names
    column_names = recode.columns.tolist()

    # Drop the first 6 columns to keep the variants 
    variants = column_names[6:]

    print(f'Number of nonsynonymous variants in {ancestry} for RAB32: {len(variants)}')
    
    # Pre-filter the dataset
    cases_data = recode[recode['PHENOTYPE'] == 2]
    controls_data = recode[recode['PHENOTYPE'] == 1]

    results = []

    for variant in variants:
        # For cases
        hom_cases = cases_data[cases_data[variant] == 2].shape[0]
        het_cases = cases_data[cases_data[variant] == 1].shape[0]
        total_cases = cases_data.shape[0]
        #freq_cases = (hom_cases + het_cases) / total_cases

        # For controls
        hom_controls = controls_data[controls_data[variant] == 2].shape[0]
        het_controls = controls_data[controls_data[variant] == 1].shape[0]
        total_controls = controls_data.shape[0]
        #freq_controls = (hom_controls + het_controls) / total_controls

        results.append({
            'Variant': variant,
            'Hom Cases': hom_cases,
            'Het Cases': het_cases,
            'Total Cases': total_cases,
            #'Carrier freq in Cases': freq_cases,
            'Hom Controls': hom_controls,
            'Het Controls': het_controls,
            'Total Controls': total_controls,
            #'Carrier freq in Controls': freq_controls
        })

    # Return
    df_results = pd.DataFrame(results)
    df_results['ID'] = df_results['Variant'].apply(lambda x: x.rsplit('_', 1)[0])
    
    # Print dimensions of the df_results dataframe
    print(f'df_results shape: {df_results.shape}')
    
    # Convert to CSV
    df_results.to_csv(f'{WORK_DIR}/{ancestry}_RAB32.nonsynonymous-variants-adj.txt', sep = '\t', index=False)

In [ ]:
ancestries_nonsyn = ['AAC','AFR','AJ','AMR','CAS','EAS','EUR','FIN','MDE','SAS','CAH']
for ancestry in ancestries_nonsyn:
    WORK_DIR = f'/home/jupyter/results/RAB32/RAB32_WGS_{ancestry}'
    input_file = f"{WORK_DIR}/{ancestry}_RAB32.nonsynonymous.nonadj.raw"
    output_file = f"{WORK_DIR}/{ancestry}_RAB32.nonsynonymous-variants.txt"

    with open(input_file) as f:
        header = f.readline().strip().split()

    # Extract columns 7+
    variant_cols = header[6:]

    # Write to output file with header
    with open(output_file, 'w') as out:
        out.write("Variants\n")
        for col in variant_cols:
            out.write(col + '\n')

In [ ]:
# Filter annotation file for missense variants
ancestries_nonsyn = ['AAC','AFR','AJ','AMR','CAS','EAS','EUR','FIN','MDE','SAS','CAH']

# Define the columns to keep in the final output
columns_to_keep = ["Chr", "Start", "End", "Ref", "Alt", "Func.refGene", "Gene.refGene", "GeneDetail.refGene", "ExonicFunc.refGene", "AAChange.refGene", "avsnp151", "SIFT_pred", "Polyphen2_HDIV_pred", "CADD_phred", "gnomad41_genome_AF"]

for ancestry in ancestries_nonsyn:

    WORK_DIR = f'/home/jupyter/results/RAB32/RAB32_WGS_{ancestry}'
    
    # Define file paths based on ancestry
    variants_file = f"{WORK_DIR}/{ancestry}_RAB32.nonsynonymous-variants.txt"
    annovar_file = f"{WORK_DIR}/{ancestry}.RAB32.annovar.hg38_multianno.txt"
    output_file = f"{WORK_DIR}/{ancestry}.RAB32_missense_results.txt"

    # Load the first file (variants file)
    variants_df = pd.read_csv(variants_file, sep="\t")
    
    # Split on '_' into two parts
    split_cols = variants_df['Variants'].str.split('_', expand=True)
    split_cols.columns = ['chr_pos_alleles', 'DominantAllele']
    
    # Now split the first part on ':'
    split_chr = split_cols['chr_pos_alleles'].str.split(':', expand=True)
    split_chr.columns = ['Chromosome', 'Position', 'Allele1', 'Allele2']
    
    # Combine into the DataFrame
    variants_df = pd.concat([variants_df, split_chr, split_cols[['DominantAllele']]], axis=1)
    
    # Convert Position to numeric
    variants_df['Position'] = pd.to_numeric(variants_df['Position'], errors='coerce')

    # Load the annovar results file
    annovar_df = pd.read_csv(annovar_file, sep="\t")

    # Convert Start to numeric for correct comparison
    annovar_df['Start'] = pd.to_numeric(annovar_df['Start'], errors='coerce')

    # Merge both dataframes on Position (from variants) and Start (from annovar)
    merged_df = annovar_df.merge(
        variants_df, 
        left_on='Start', 
        right_on='Position', 
        how='inner'
    )

    # Filter where Ref matches Allele1 and Alt match Allele2
    filtered_df = merged_df[(merged_df['Ref'] == merged_df['Allele1']) & 
                            (merged_df['Alt'] == merged_df['Allele2'])]

    # Save the filtered results to a new file
    filtered_df.to_csv(output_file, sep="\t", index=False)
    
    # Keep only specific columns
    filtered_df = filtered_df[columns_to_keep]

    # Save the filtered results to a new file
    filtered_df.to_csv(output_file, sep="\t", index=False)

In [ ]:
# Filter annotation file for rare missense variants with a gnomAD AF <0.0001
ancestries_nonsyn = ['AAC','AFR','AJ','AMR','CAS','EAS','EUR','FIN','MDE','SAS','CAH']

for ancestry in ancestries_nonsyn:

    WORK_DIR = f'/home/jupyter/results/RAB32/RAB32_WGS_{ancestry}'

    # Define file paths based on ancestry
    missense_file = f"{WORK_DIR}/{ancestry}.RAB32_missense_results.txt"
    rare_file = f"{WORK_DIR}/{ancestry}.RAB32_rare_missense_results.txt"

    # Load the file with all missense variants
    missense_df = pd.read_csv(missense_file, sep="\t")

    # Make sure all gnomad AF are numeric and keep missing gnomAD values
    missense_df["gnomad41_genome_AF"] = missense_df["gnomad41_genome_AF"].replace(["", ".", "NA", "", ""], pd.NA)
    gnomAD_numeric = pd.to_numeric(missense_df["gnomad41_genome_AF"], errors="coerce")

    # Filter for rare variants with gnomadAF <0.0001 and apply filter
    mask_keep = (gnomAD_numeric < 0.0001) | gnomAD_numeric.isna()
    rare_df = missense_df[mask_keep]

    # Save filtered results
    rare_df.to_csv(rare_file, sep="\t", index=False)

In [ ]:
# Create one file with all rare missense variants in all ancestries
ancestries_nonsyn = ['AAC','AFR','AJ','AMR','CAS','EAS','EUR','FIN','MDE','SAS','CAH']

all_rare_missense_dfs = []

for ancestry in ancestries_nonsyn:
    WORK_DIR = f'/home/jupyter/results/RAB32/RAB32_WGS_{ancestry}'

    # Define file paths based on ancestry
    rare_missense_file = f"{WORK_DIR}/{ancestry}.RAB32_rare_missense_results.txt"

    # Load the file with all rare missense variants
    rare_missense_df = pd.read_csv(rare_missense_file, sep="\t")

    # Skip empty/all-NA files to avoid the concat warning
    if rare_missense_df.empty or rare_missense_df.dropna(how='all').empty:
        print('Empty or all-NA DataFrame for ancestry', ancestry, '- skipping')
        continue

    # Add ancestry column
    rare_missense_df.insert(0, 'ancestry', ancestry)

    all_rare_missense_dfs.append(rare_missense_df)

# Merge all ancestries into one table
merged_df = pd.concat(all_rare_missense_dfs, ignore_index=True)

# Save merged file
output_path = '/home/jupyter/results/RAB32/RAB32_WGS_all_rare_missense_variants.txt'
merged_df.to_csv(output_path, sep='\t', index=False)

print('Merged file written to:', output_path)

### Look into *RAB32* variants of interest

In [ ]:
# Prepare a single file with all RAB32 rare missense variants
RESULTS_DIR = Path("/home/jupyter/results/RAB32")
PLINK = "/home/jupyter/tools/plink/plink1.9"

VARIANT_FILE = RESULTS_DIR / "RAB32_WGS_all_rare_missense_variants.txt"

TEMP_DIR = RESULTS_DIR  / "subprocess"
TEMP_DIR.mkdir(parents=True, exist_ok=True)

# Read the filtered variant file
variants_df = pd.read_csv(VARIANT_FILE, sep="\t", dtype=str)
variants_df.columns = variants_df.columns.str.strip()

# Remove accidental whitespace
for col in variants_df.columns:
    variants_df[col] = variants_df[col].str.strip()

# Check the input
print("Number of variant records:", len(variants_df))

ancestry_to_folder = {
    ancestry: ancestry
    for ancestry in variants_df["ancestry"].dropna().unique()
}

In [ ]:
# Prepare variant identifiers (Chr:Pos:Ref:Alt)
def make_variant_id(row):
    """Construct a genomic variant ID from the variant table."""

    # Extract the chromosome and remove surrounding whitespace
    chrom = str(row["Chr"]).strip()

    # Add the 'chr' prefix if it is not already present
    if not chrom.lower().startswith("chr"):
        chrom = "chr" + chrom

    # Extract the genomic position.
    position = str(row["Start"]).strip()

    # Extract the reference and alternate alleles
    ref = str(row["Ref"]).strip()
    alt = str(row["Alt"]).strip()

    # Combine the four components into a single variant ID
    return f"{chrom}:{position}:{ref}:{alt}"

# Apply the function to every row in the variant table
variants_df["variant_id"] = variants_df.apply(make_variant_id, axis=1)

# Create a descriptive label for each variant (AAChange.refGene)
def make_variant_label(row):
    aa_change = str(row["AAChange.refGene"]).strip()

    # Check whether the annotation contains a usable value
    if aa_change and aa_change.lower() not in {
        "nan", "none", "."
    }:
        return aa_change

    # Use the genomic ID if the protein annotation is missing
    return row["variant_id"]

variants_df["variant_label"] = variants_df.apply(make_variant_label, axis=1)

In [ ]:
# Extract variant carriers using PLINK
all_carrier_records = []

# Process each ancestry separately
for ancestry, ancestry_variants in variants_df.groupby(
    "ancestry",
    sort=True
):

    # Translate the ancestry label into the PLINK folder name
    folder_name = ancestry_to_folder[ancestry]

    # Define the folder and prefix for the ancestry-specific PLINK binary files
    work_dir = RESULTS_DIR / f"RAB32_WGS_{folder_name}"
    bfile_prefix = work_dir / "RAB32"

    print(f"\nProcessing ancestry: {ancestry}")
    print(f"PLINK dataset: {bfile_prefix}")

    # Check that the PLINK binary files exist
    required_files = [Path(str(bfile_prefix) + ".bed"), Path(str(bfile_prefix) + ".bim"), Path(str(bfile_prefix) + ".fam")]

    # Collect the paths of any files that are missing
    missing_files = [
        str(path)
        for path in required_files
        if not path.exists()
    ]

    # If any files are missing, record the problem and skip this ancestry 
    if missing_files:
        print("ERROR: Missing PLINK files:")
        for path in missing_files:
            print("  ", path)
        continue

    # Read the BIM file to check variant IDs and alleles
    bim = pd.read_csv(str(bfile_prefix) + ".bim", sep=r"\s+", header=None, dtype=str)

    # Create a set of all variant IDs for fast membership checks
    bim_ids = set(bim[1])

    # Build a dictionary mapping each variant ID to its two alleles as recorded in the PLINK BIM file
    bim_alleles = {
        variant_id: {a1.upper(), a2.upper()}
        for variant_id, a1, a2 in zip(
            bim[1], bim[4], bim[5]
        )
    }

    # Process every variant assigned to this ancestry
    for i, (_, row) in enumerate(
        ancestry_variants.iterrows(),
        start=1
    ):

        # Extract the identifiers and alternate allele
        variant_id = row["variant_id"]
        variant_label = row["variant_label"]
        alt = str(row["Alt"]).strip().upper()

        print(f"  Variant: {variant_id}")

        # Check whether the variant exists in the BIM file
        if variant_id not in bim_ids:
            print("    SKIPPED: Variant ID not found in BIM.")
            continue

        # Check whether the alternate allele is present
        if alt not in bim_alleles[variant_id]:
            print("    SKIPPED: Alternate allele mismatch.")
            continue

        # Define output filenames for this variant
        safe_ancestry = re.sub(
            r"[^A-Za-z0-9_-]",
            "_",
            ancestry
        )

        # Prefix for PLINK output files, including the .raw file
        raw_prefix = TEMP_DIR/(f"{safe_ancestry}.{i:04d}.RAB29_variant")

        # PLINK's allele file tells it which allele to count
        allele_file = TEMP_DIR/(f"{safe_ancestry}.{i:04d}.allele.txt")

        # Write the variant ID and alternate allele to a temporary text file for PLINK
        pd.DataFrame([[variant_id, alt]]).to_csv(allele_file, sep="\t", index=False, header=False)

        # Construct and execute the PLINK command
        command = [
            PLINK,
            "--bfile", str(bfile_prefix),
            "--snp", variant_id,
            "--keep-allele-order",
            "--recode", "A",
            "--recode-allele", str(allele_file),
            "--out", str(raw_prefix),
        ]

        # Run PLINK and capture its output so that errors can be displayed if the command fails
        result = subprocess.run(
            command,
            capture_output=True,
            text=True,
        )

        # The expected genotype output file
        raw_file = Path(str(raw_prefix) + ".raw")

        # Check whether PLINK completed successfully
        if result.returncode != 0 or not raw_file.exists():
            print("    ERROR: PLINK failed.")
            print(result.stderr[-1500:])
            continue

        # Read the genotype file and identify carriers
        raw_df = pd.read_csv(raw_file, sep=r"\s+", dtype=str)

        # Remove any accidental whitespace in column names
        raw_df.columns = raw_df.columns.str.strip()

        # Check that the file has the expected structure
        if "IID" not in raw_df.columns or raw_df.shape[1] < 7:
            print("    ERROR: Unexpected .raw file format.")
            continue


        # Identify the dosage column and convert its values from text to numeric values
        genotype_col = raw_df.columns[-1]
        dosage = pd.to_numeric(raw_df[genotype_col], errors="coerce")
        carrier_df = raw_df.loc[dosage.gt(0), ["FID", "IID"]].copy()

        # Extract the sample IDs used for matching with the GP2ID column in the clinical data
        carrier_ids = carrier_df["IID"].astype(str).tolist()
        print(f"    Number of carriers: {len(carrier_ids)}")

        # Store the carrier IDs in the master carrier table
        for iid in carrier_ids:
            all_carrier_records.append({
                "ancestry": ancestry,
                "variant_id": variant_id,
                "variant_label": variant_label,
                "GP2ID": iid,
            })
        
        # Remove intermediate PLINK files
        for file in TEMP_DIR.glob(f"{safe_ancestry}.{i:04d}.*"):
            file.unlink()

# Save the carrier table and summary
carriers_df = pd.DataFrame(
    all_carrier_records,
    columns=[
        "ancestry",
        "variant_id",
        "variant_label",
        "GP2ID",
    ],
)

# Remove temporary directory and any remaining intermediate files
shutil.rmtree(TEMP_DIR, ignore_errors=True)

# Save all carrier IDs across all ancestries and variants.
carriers_df.to_csv(RESULTS_DIR / "RAB32_all_variant_carriers.csv", index=False,)

In [ ]:
# Merge carrier info with clinical data

clinical_data_df_pheno = pd.read_csv(f'/home/jupyter/results/Clinical_master_key_new_R12.csv', index=False)

# Prepare the clinical and carrier tables for merging
clinical_df = clinical_data_df_pheno.copy()

# Convert both ID columns to strings and remove whitespace
clinical_df["GP2ID"] = (clinical_df["GP2ID"].astype("string").str.strip())
carriers_df["GP2ID"] = (carriers_df["GP2ID"].astype("string").str.strip())

# Exclude empty carrier IDs before matching.
carriers_df = carriers_df[carriers_df["GP2ID"].notna() & carriers_df["GP2ID"].ne("")].copy()

# Rename the ancestry column before merging
carriers_for_merge = carriers_df.rename(columns={"ancestry": "genetic_ancestry"})

# Merge the carrier table and clinical data
merged_clinical_df = carriers_for_merge.merge(clinical_df, on="GP2ID", how="left", indicator=True, validate="many_to_one")

# Check the matching results
print("Total carrier-variant records:", len(merged_clinical_df))
print("Records matched to clinical data:", (merged_clinical_df["_merge"] == "both").sum())
print("Records without a clinical match:", (merged_clinical_df["_merge"] == "left_only").sum())

# Save the merged data
merged_clinical_df = merged_clinical_df.drop(columns="_merge")

# Save the combined clinical data for all ancestries.
merged_clinical_df.to_csv("/home/jupyter/results/RAB32/RAB32_all_ancestries_merged_clinical.csv", index=False,)

In [ ]:
# Save ancestry-specific clinical files
for ancestry, df in merged_clinical_df.groupby("genetic_ancestry", sort=True):

    # Replace spaces or special characters in the ancestry label to create a safe output filename.
    safe_ancestry = re.sub(r"[^A-Za-z0-9_-]", "_", ancestry)

    # Construct the output filename for this ancestry
    # Define a separate directory for the ancestry-specific clinical files
    CLINICAL_OUTPUT_DIR = Path("/home/jupyter/results/RAB32")
    output_file = CLINICAL_OUTPUT_DIR/(f"RAB32_{safe_ancestry}_merged_clinical.csv")

    # Save all carrier-variant records for this ancestry
    df.to_csv(output_file, index=False)

    print(f"Saved {ancestry}: {len(df)} carrier-variant records")

In [ ]:
# Count carriers per variant and phenotype

# Define the input and output files
merged_file = (CLINICAL_OUTPUT_DIR/"RAB32_all_ancestries_merged_clinical.csv")
output_file = (CLINICAL_OUTPUT_DIR/"RAB32_all_ancestries_merged_clinical_with_counts.csv")

# Read the merged clinical data
merged_df = pd.read_csv(merged_file, dtype={"GP2ID": str})

# Check that the required columns are present
required_columns = ["variant_id", "GP2ID", "new_assigned_phenotype"]

missing_columns = [
    col for col in required_columns
    if col not in merged_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

# Count unique carriers for each variant and phenotype
phenotype_counts = (merged_df.groupby(["variant_id", "new_assigned_phenotype"])["GP2ID"].nunique().unstack(fill_value=0))

# Ensure all three phenotype groups have a column
phenotype_groups = ["PD", "Control", "Others"]

for phenotype in phenotype_groups:
    if phenotype not in phenotype_counts.columns:
        phenotype_counts[phenotype] = 0

# Rename the columns to make their meaning explicit
phenotype_counts = phenotype_counts[phenotype_groups].rename(columns={
    "PD": "RAB32_variant_PD_count",
    "Control": "RAB32_variant_Control_count",
    "Others": "RAB32_variant_Others_count"
})

# Add the counts to the original merged clinical data
merged_df = merged_df.merge(
    phenotype_counts,
    left_on="variant_id",
    right_index=True,
    how="left",
    validate="many_to_one"
)

#  Save the updated file
merged_df.to_csv(output_file, index=False)